
# 💬 NB05 — Contextual Sentiment & Financial Signals
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Modeling

| | |
|---|---|
| **Input** | `data/silver/silver_articles.parquet` |
| **Output** | `data/silver/silver_enriched.parquet` · `data/gold/sentiment/articles_sentiment.parquet` |
| **Engines** | Léxico FII PT-BR customizado + TextBlob fallback |

## 🎯 O que este notebook faz

1. **Léxico FII PT-BR** — dicionário de 70+ termos com score `[-1, +1]`.
2. **Sentimento por documento** — `polarity_score` numérico + `sentiment_label` categórico.
3. **Sinais contextuais** — flags binárias e scores de intensidade por categoria:
   - `flag_risco`, `flag_oportunidade`, `flag_dividendo`, `flag_inadimplencia`, `flag_vacancia`, `flag_crise`
4. **Dataset Silver Enriched** — Silver + 12 colunas de sentimento e sinais.
5. **Gold Sentiment** — tabela analítica com stats por fonte, período e categoria.

## ⚠️ Por que NÃO usar VADER/TextBlob puro para FIIs?

| Ferramenta | Problema com FIIs |
|---|---|
| VADER (EN) | "dividend" = neutro; "yield" = positivo por contexto errado |
| TextBlob (PT) | "vacância" = positivo (inglês: vacancy = oportunidade) |
| **Léxico FII PT-BR** | Termos calibrados para contexto financeiro imobiliário BR |

> **LGPD / Responsible AI:** Nenhum dado pessoal é analisado.
> Sentimento aplica-se exclusivamente a conteúdo editorial/social sobre FIIs.

## 📦 Seção 1 — Imports

In [1]:

import sys, os, warnings
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, random, unicodedata, json
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import nltk

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType, IntegerType, BooleanType
)
import pyspark

try:
    from textblob import TextBlob
    TEXTBLOB_OK = True
except ImportError:
    TEXTBLOB_OK = False

warnings.filterwarnings("ignore")
print(f"Python   : {sys.version.split()[0]}")
print(f"PySpark  : {pyspark.__version__}")
print(f"TextBlob : {'OK' if TEXTBLOB_OK else 'NOT AVAILABLE — using lexicon only'}")

Python   : 3.12.12
PySpark  : 4.1.2
TextBlob : OK


## ⚙️ Seção 2 — Configuração e SparkSession

In [2]:

PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent; _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import (
    RANDOM_SEED, SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SILVER_DIR   = PROJECT_ROOT / "data" / "silver"
GOLD_SENT_DIR = PROJECT_ROOT / "data" / "gold" / "sentiment"
GOLD_SENT_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_sentiment")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
log_spark_start(logger, spark.sparkContext.appName, spark.version)

for _res in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"corpora/{_res}")
    except LookupError:
        try: nltk.download(_res, quiet=True)
        except: pass
try:
    from nltk.corpus import stopwords as _nltk_sw
    STOPWORDS_PT = set(_nltk_sw.words("portuguese"))
except Exception:
    STOPWORDS_PT = set()

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"GOLD_SENT_DIR   : {GOLD_SENT_DIR}")
print(f"Spark           : {spark.sparkContext.appName} | {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:49:21 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:49:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:49:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 15:49:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/14 15:49:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/14 15:49:23 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


2026-06-14T15:49:25 | INFO     | fii_pipeline.ingestion | SPARK_START | app=FIIIntelligencePlatform_sentiment | spark=4.1.2
PROJECT_ROOT    : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
GOLD_SENT_DIR   : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/sentiment
Spark           : FIIIntelligencePlatform_sentiment | 4.1.2



## 📚 Seção 3 — Léxico FII PT-BR Customizado

**Calibração:** termos revisados para contexto financeiro imobiliário brasileiro.
Scores em `[-1.0, +1.0]`. `0.0` = neutro.

**Categorias de sinais:**

| Categoria | Sinais | Impacto |
|-----------|--------|---------|
| `dividendo` | dividendo, provento, yield, rendimento | ++ Marketing positivo |
| `oportunidade` | compra, desconto, pvp, barato | + Intenção de compra |
| `risco` | risco, volatilidade, incerteza | - Alerta |
| `crise` | crise, crash, colapso, queda | -- Urgência negativa |
| `vacancia` | vacância, desocupação, vacante | -- Fundamental negativo |
| `inadimplencia` | inadimplência, calote, default | -- Fundamental crítico |

In [3]:

# ── Léxico FII PT-BR calibrado ────────────────────────────────────────────────
FII_LEXICON: Dict[str, float] = {
    # Positivos fortes (+0.7 a +1.0)
    "dividendo":      +0.85, "dividendos":      +0.85,
    "provento":       +0.80, "proventos":       +0.80,
    "rendimento":     +0.75, "rendimentos":     +0.75,
    "distribuicao":   +0.70, "distribuicoes":   +0.70,
    "lucro":          +0.75, "lucros":          +0.75,
    "crescimento":    +0.70, "valorizacao":     +0.70,
    "oportunidade":   +0.75, "oportunidades":   +0.75,
    "recomendado":    +0.65, "positivo":        +0.60,
    "compra":         +0.65, "comprar":         +0.65,
    "aumento":        +0.55, "expansao":        +0.60,
    "recuperacao":    +0.65, "retomada":        +0.60,
    "alta":           +0.50, "subida":          +0.55,
    "robusto":        +0.65, "sólido":          +0.65,
    "record":         +0.80, "historico":       +0.60,
    "melhor":         +0.60, "excelente":       +0.80,
    "forte":          +0.55, "sustentavel":     +0.65,
    "desconto":       +0.60, "barato":          +0.55,
    "pvp":            +0.40, "yield":           +0.70,
    "renda":          +0.55, "passiva":         +0.55,

    # Negativos fortes (-0.7 a -1.0)
    "crise":         -0.85,  "crash":          -0.90,
    "colapso":       -0.95,  "falencia":       -0.90,
    "inadimplencia": -0.90,  "calote":         -0.90,
    "default":       -0.85,  "prejuizo":       -0.80,
    "perda":         -0.70,  "perdas":         -0.70,
    "queda":         -0.65,  "quedas":         -0.65,
    "reducao":       -0.55,  "corte":          -0.60,
    "vacancia":      -0.75,  "vacante":        -0.70,
    "desocupacao":   -0.75,  "desocupado":     -0.70,
    "risco":         -0.55,  "riscos":         -0.55,
    "volatilidade":  -0.45,  "incerteza":      -0.50,
    "negativo":      -0.70,  "pessimista":     -0.65,
    "venda":         -0.40,  "vender":         -0.40,
    "baixo":         -0.40,  "fraco":          -0.50,
    "fraqueza":      -0.55,  "deterioracao":   -0.70,
    "desvalorizacao":-0.70,  "depreciacao":    -0.65,
    "problema":      -0.55,  "problemas":      -0.55,
    "preocupante":   -0.60,  "alerta":         -0.45,
    "critico":       -0.65,  "grave":          -0.70,

    # Neutros relevantes
    "analise":     0.0, "fundo": 0.0, "fii": 0.0, "cota": 0.0,
    "cotacao":     0.0, "mercado": 0.0, "setor": 0.0, "carteira": 0.0,
}

# ── Termos de sinal por categoria ────────────────────────────────────────────
SIGNAL_TERMS: Dict[str, List[str]] = {
    "dividendo":     ["dividendo","dividendos","provento","proventos","rendimento",
                      "rendimentos","distribuicao","yield","renda","income"],
    "oportunidade":  ["oportunidade","compra","desconto","barato","pvp","valorizacao",
                      "recomendado","crescimento","expansao","lucro"],
    "risco":         ["risco","riscos","volatilidade","incerteza","alerta","cautela",
                      "negativo","deterioracao","problema","fraco"],
    "crise":         ["crise","crash","colapso","queda","quedas","desvalorizacao",
                      "falencia","prejuizo","perda","grave","critico"],
    "vacancia":      ["vacancia","vacante","desocupacao","desocupado","vago","vacante"],
    "inadimplencia": ["inadimplencia","calote","default","atraso","nao_pagamento"],
}

print(f"✅ Léxico FII PT-BR: {len(FII_LEXICON):,} termos calibrados")
print(f"   Positivos  : {sum(1 for v in FII_LEXICON.values() if v > 0)}")
print(f"   Negativos  : {sum(1 for v in FII_LEXICON.values() if v < 0)}")
print(f"   Neutros    : {sum(1 for v in FII_LEXICON.values() if v == 0)}")
print(f"\n   Categorias de sinal: {list(SIGNAL_TERMS.keys())}")

✅ Léxico FII PT-BR: 84 termos calibrados
   Positivos  : 38
   Negativos  : 38
   Neutros    : 8

   Categorias de sinal: ['dividendo', 'oportunidade', 'risco', 'crise', 'vacancia', 'inadimplencia']


## 🧮 Seção 4 — Funções de Análise de Sentimento

In [4]:

_TOKEN_RE = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)

def _norm(s: str) -> str:
    return unicodedata.normalize("NFD", s).encode("ascii","ignore").decode("ascii").lower()

def tokenize_pt(text: str) -> List[str]:
    if not text or not isinstance(text, str): return []
    return [_norm(t) for t in _TOKEN_RE.findall(text.lower())]

def sentiment_score_lexicon(text: str) -> Tuple[float, int, int]:
    """Retorna (polarity, n_pos_terms, n_neg_terms) via léxico FII PT-BR."""
    tokens = tokenize_pt(text)
    if not tokens:
        return 0.0, 0, 0
    scores   = [FII_LEXICON[t] for t in tokens if t in FII_LEXICON]
    n_pos    = sum(1 for s in scores if s > 0)
    n_neg    = sum(1 for s in scores if s < 0)
    polarity = float(np.mean(scores)) if scores else 0.0
    return round(polarity, 4), n_pos, n_neg

def sentiment_label(score: float) -> str:
    """Converte score em rótulo categórico PT."""
    if score >=  0.15: return "positivo"
    if score <= -0.15: return "negativo"
    return "neutro"

def compute_signals(text: str) -> Dict[str, object]:
    """Retorna flags e scores de intensidade para cada categoria de sinal."""
    tokens_set = set(tokenize_pt(text))
    result: Dict[str, object] = {}
    for cat, terms in SIGNAL_TERMS.items():
        matched = [t for t in terms if t in tokens_set]
        result[f"flag_{cat}"]  = len(matched) > 0
        result[f"score_{cat}"] = round(min(len(matched) / max(len(terms), 1), 1.0), 4)
    return result

def analyze_document(text: str) -> Dict[str, object]:
    """Combinação completa: sentimento + sinais."""
    polarity, n_pos, n_neg = sentiment_score_lexicon(text)
    signals = compute_signals(text)

    # TextBlob fallback para documentos sem termos do léxico
    tb_polarity = None
    if polarity == 0.0 and TEXTBLOB_OK:
        try:
            tb = TextBlob(text[:500])
            tb_polarity = round(float(tb.sentiment.polarity), 4)
            if abs(tb_polarity) > abs(polarity):
                polarity = tb_polarity
        except Exception:
            pass

    return {
        "polarity_score":     polarity,
        "sentiment_label":    sentiment_label(polarity),
        "n_pos_terms":        n_pos,
        "n_neg_terms":        n_neg,
        "textblob_polarity":  tb_polarity,
        **signals,
    }

# Smoke test
_test_texts = [
    "O fundo imobiliário pagou excelentes dividendos e proventos recordes este trimestre.",
    "Vacância alta e inadimplência crítica causam queda abrupta nos rendimentos.",
    "Análise do portfólio de FIIs de tijolo no setor de galpões logísticos.",
]
for t in _test_texts:
    r = analyze_document(t)
    print(f"  [{r['sentiment_label']:8}] ({r['polarity_score']:+.3f}) → {t[:65]}...")

  [positivo] (+0.550) → O fundo imobiliário pagou excelentes dividendos e proventos recor...
  [negativo] (-0.210) → Vacância alta e inadimplência crítica causam queda abrupta nos re...
  [neutro  ] (+0.000) → Análise do portfólio de FIIs de tijolo no setor de galpões logíst...


## ⚡ Seção 5 — Análise Spark: Silver Enriched

In [5]:

SILVER_PATH = SILVER_DIR / "silver_articles.parquet"
if not SILVER_PATH.exists():
    raise FileNotFoundError(f"Silver não encontrado: {SILVER_PATH}\nExecute NB02 primeiro.")

df = pd.read_parquet(SILVER_PATH)
df["text_clean"] = df.get("text_clean", pd.Series([""] * len(df))).fillna("").astype(str)
print(f"Silver carregado: {len(df):,} documentos")

# ── Aplicar análise via Spark UDF ─────────────────────────────────────────────
from pyspark.sql.types import (StructType, StructField, FloatType,
                                BooleanType, StringType, IntegerType)

SENTIMENT_SCHEMA = StructType([
    StructField("polarity_score",    FloatType(),   True),
    StructField("sentiment_label",   StringType(),  True),
    StructField("n_pos_terms",       IntegerType(), True),
    StructField("n_neg_terms",       IntegerType(), True),
    StructField("textblob_polarity", FloatType(),   True),
    StructField("flag_dividendo",    BooleanType(), True),
    StructField("score_dividendo",   FloatType(),   True),
    StructField("flag_oportunidade", BooleanType(), True),
    StructField("score_oportunidade",FloatType(),   True),
    StructField("flag_risco",        BooleanType(), True),
    StructField("score_risco",       FloatType(),   True),
    StructField("flag_crise",        BooleanType(), True),
    StructField("score_crise",       FloatType(),   True),
    StructField("flag_vacancia",     BooleanType(), True),
    StructField("score_vacancia",    FloatType(),   True),
    StructField("flag_inadimplencia",BooleanType(), True),
    StructField("score_inadimplencia",FloatType(),  True),
])

@F.udf(returnType=SENTIMENT_SCHEMA)
def analyze_udf(text: str):
    import re, unicodedata, numpy as np
    from typing import Dict, List, Tuple
    _TOKEN_RE2 = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)
    def _n(s):
        return unicodedata.normalize("NFD", s).encode("ascii","ignore").decode("ascii").lower()
    def _tok(t):
        if not t: return []
        return [_n(w) for w in _TOKEN_RE2.findall(t.lower())]

    # Inline léxico compacto (broadcast-safe)
    LEX = {
        "dividendo":+0.85,"dividendos":+0.85,"provento":+0.80,"proventos":+0.80,
        "rendimento":+0.75,"rendimentos":+0.75,"lucro":+0.75,"crescimento":+0.70,
        "oportunidade":+0.75,"recomendado":+0.65,"compra":+0.65,"alta":+0.50,
        "record":+0.80,"excelente":+0.80,"forte":+0.55,"desconto":+0.60,
        "yield":+0.70,"renda":+0.55,"valorizacao":+0.70,"retomada":+0.60,
        "crise":-0.85,"crash":-0.90,"colapso":-0.95,"falencia":-0.90,
        "inadimplencia":-0.90,"calote":-0.90,"default":-0.85,"prejuizo":-0.80,
        "perda":-0.70,"queda":-0.65,"quedas":-0.65,"reducao":-0.55,"corte":-0.60,
        "vacancia":-0.75,"vacante":-0.70,"desocupacao":-0.75,"risco":-0.55,
        "volatilidade":-0.45,"incerteza":-0.50,"negativo":-0.70,"venda":-0.40,
        "baixo":-0.40,"fraco":-0.50,"deterioracao":-0.70,"desvalorizacao":-0.70,
    }
    SIG = {
        "dividendo":["dividendo","dividendos","provento","proventos","rendimento","yield","renda"],
        "oportunidade":["oportunidade","compra","desconto","barato","valorizacao","crescimento"],
        "risco":["risco","riscos","volatilidade","incerteza","negativo","deterioracao","fraco"],
        "crise":["crise","crash","colapso","queda","quedas","desvalorizacao","falencia","prejuizo"],
        "vacancia":["vacancia","vacante","desocupacao","desocupado"],
        "inadimplencia":["inadimplencia","calote","default"],
    }
    tokens = _tok(text or "")
    scores = [LEX[t] for t in tokens if t in LEX]
    n_pos  = sum(1 for s in scores if s > 0)
    n_neg  = sum(1 for s in scores if s < 0)
    pol    = float(np.mean(scores)) if scores else 0.0
    lbl    = "positivo" if pol >= 0.15 else ("negativo" if pol <= -0.15 else "neutro")
    tset   = set(tokens)
    row    = [round(pol,4), lbl, n_pos, n_neg, None]
    for cat in ["dividendo","oportunidade","risco","crise","vacancia","inadimplencia"]:
        m = [t for t in SIG.get(cat,[]) if t in tset]
        row += [len(m)>0, round(min(len(m)/max(len(SIG[cat]),1),1.0),4)]
    return tuple(row)

sdf = spark.createDataFrame(df.fillna(""))
sdf_sent = sdf.withColumn("_s", analyze_udf(F.col("text_clean")))

for field in SENTIMENT_SCHEMA.fields:
    sdf_sent = sdf_sent.withColumn(field.name, F.col(f"_s.{field.name}"))
sdf_sent = sdf_sent.drop("_s")

print(f"✅ Análise de sentimento aplicada via Spark UDF")
print(f"   Total de registros: {sdf_sent.count():,}")

Silver carregado: 126 documentos
✅ Análise de sentimento aplicada via Spark UDF


   Total de registros: 126


## 💾 Seção 6 — Salvar Silver Enriched + Gold Sentiment

In [6]:

# ── Silver Enriched ──────────────────────────────────────────────────────────
df_enriched = sdf_sent.toPandas()

enriched_path = SILVER_DIR / "silver_enriched.parquet"
df_enriched.to_parquet(enriched_path, index=False, compression="snappy")
print(f"✅ silver_enriched.parquet: {len(df_enriched):,} docs × {len(df_enriched.columns)} cols")

# ── Distribuição de sentimento ────────────────────────────────────────────────
sent_dist = df_enriched["sentiment_label"].value_counts()
print(f"\n   Distribuição de sentimento:")
for lbl, cnt in sent_dist.items():
    pct = cnt / len(df_enriched) * 100
    print(f"   {lbl:12} : {cnt:>6,} ({pct:5.1f}%)")

print(f"\n   Média polarity_score : {df_enriched['polarity_score'].mean():.4f}")
print(f"   Flags ativas:")
for cat in ["dividendo","oportunidade","risco","crise","vacancia","inadimplencia"]:
    n = df_enriched.get(f"flag_{cat}", pd.Series(dtype=bool)).sum()
    print(f"     flag_{cat:<16} : {n:>6,} docs")

26/06/14 15:49:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ silver_enriched.parquet: 126 docs × 37 cols

   Distribuição de sentimento:
   positivo     :     70 ( 55.6%)
   neutro       :     39 ( 31.0%)
   negativo     :     17 ( 13.5%)

   Média polarity_score : 0.2475
   Flags ativas:
     flag_dividendo        :     64 docs
     flag_oportunidade     :     40 docs
     flag_risco            :     48 docs
     flag_crise            :     33 docs
     flag_vacancia         :      2 docs
     flag_inadimplencia    :     10 docs


In [7]:

# ── Gold Sentiment: stats por fonte + período ─────────────────────────────────
_df = df_enriched.copy()
_df["pub_month"] = pd.to_datetime(
    _df.get("published_dt", _df.get("collected_at", pd.NaT)), errors="coerce"
).dt.to_period("M").astype(str)

# Stats por fonte
gold_by_source = (
    _df.groupby("source_label" if "source_label" in _df.columns else "source")
    .agg(
        n_articles       = ("article_id",      "count"),
        avg_polarity     = ("polarity_score",   "mean"),
        pct_positivo     = ("sentiment_label",  lambda x: (x=="positivo").mean()*100),
        pct_negativo     = ("sentiment_label",  lambda x: (x=="negativo").mean()*100),
        pct_neutro       = ("sentiment_label",  lambda x: (x=="neutro").mean()*100),
        n_dividendo      = ("flag_dividendo",   "sum"),
        n_risco          = ("flag_risco",       "sum"),
        n_crise          = ("flag_crise",       "sum"),
    )
    .round(3)
    .reset_index()
)

_sent_source_path = GOLD_SENT_DIR / "sentiment_by_source.parquet"
gold_by_source.to_parquet(_sent_source_path, index=False, compression="snappy")
print(f"✅ sentiment_by_source.parquet: {len(gold_by_source)} fontes")
print(gold_by_source[["source_label" if "source_label" in gold_by_source.columns else "source",
                       "n_articles","avg_polarity","pct_positivo","pct_negativo"]].to_string(index=False))

# Stats por mês
if _df["pub_month"].notna().sum() > 5:
    gold_by_month = (
        _df.groupby("pub_month")
        .agg(
            n_articles   = ("article_id",    "count"),
            avg_polarity = ("polarity_score","mean"),
            pct_pos      = ("sentiment_label", lambda x: (x=="positivo").mean()*100),
            pct_neg      = ("sentiment_label", lambda x: (x=="negativo").mean()*100),
        )
        .round(3)
        .reset_index()
        .sort_values("pub_month")
    )
    _sent_month_path = GOLD_SENT_DIR / "sentiment_by_month.parquet"
    gold_by_month.to_parquet(_sent_month_path, index=False, compression="snappy")
    print(f"\n✅ sentiment_by_month.parquet: {len(gold_by_month)} meses")
    print(gold_by_month.to_string(index=False))

# Artigos com sentimento completo
_sent_articles_path = GOLD_SENT_DIR / "articles_sentiment.parquet"
_cols = ["article_id","source","source_type","source_label","title","url",
         "published_dt","collected_at","polarity_score","sentiment_label",
         "n_pos_terms","n_neg_terms","flag_dividendo","flag_oportunidade",
         "flag_risco","flag_crise","flag_vacancia","flag_inadimplencia",
         "score_dividendo","score_oportunidade","score_risco","score_crise"]
_available = [c for c in _cols if c in _df.columns]
_df[_available].to_parquet(_sent_articles_path, index=False, compression="snappy")
print(f"\n✅ articles_sentiment.parquet: {len(_df):,} docs × {len(_available)} cols")

✅ sentiment_by_source.parquet: 16 fontes
        source_label  n_articles  avg_polarity  pct_positivo  pct_negativo
  Bora Investir (B3)           4         0.347        75.000         0.000
 CNN Brasil Business          17        -0.263        17.647        58.824
           Clube FII           4         0.188        25.000         0.000
E-Investidor Estadão           2         0.076        50.000         0.000
           Empiricus          32         0.275        59.375         9.375
   Eu Quero Investir           7         0.332        57.143         0.000
        Exame Invest           1        -0.550         0.000       100.000
         FIIs.com.br          10         0.611        90.000         0.000
      Funds Explorer          12         0.347        75.000         0.000
           InfoMoney           1         0.000         0.000         0.000
        Investidor10          10         0.647       100.000         0.000
         Money Times           4         0.110        50.00

## 📊 Seção 7 — Distribuições (Plotly — validação)

In [8]:
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}]],
        subplot_titles=["Distribuição de Sentimento",
                        "Polarity Score por Fonte",
                        "Flags de Sinais FII"],
    )

    # Gráfico 1: Distribuição de sentimento (pizza)
    _sd = df_enriched["sentiment_label"].value_counts()
    fig.add_trace(go.Pie(
        labels=_sd.index.tolist(),
        values=_sd.values.tolist(),
        marker_colors=["#2ecc71","#e74c3c","#95a5a6"],
        showlegend=True,
    ), row=1, col=1)

    # Gráfico 2: Polarity por fonte (box)
    _src_col = "source_label" if "source_label" in df_enriched.columns else "source"
    _top5 = df_enriched[_src_col].value_counts().head(5).index.tolist()
    _df5  = df_enriched[df_enriched[_src_col].isin(_top5)]
    for src in _top5:
        _d = _df5[_df5[_src_col]==src]["polarity_score"].dropna()
        fig.add_trace(go.Box(y=_d.tolist(), name=src[:20], showlegend=False), row=1, col=2)

    # Gráfico 3: Flags de sinais (barras)
    _cats = ["dividendo","oportunidade","risco","crise","vacancia","inadimplencia"]
    _vals = [int(df_enriched.get(f"flag_{c}", pd.Series(dtype=bool)).sum()) for c in _cats]
    _colors = ["#2ecc71","#27ae60","#e67e22","#e74c3c","#c0392b","#8e44ad"]
    fig.add_trace(go.Bar(x=_cats, y=_vals, marker_color=_colors, showlegend=False), row=1, col=3)

    fig.update_layout(
        title_text="NB05 — Análise de Sentimento FII PT-BR",
        template="plotly_dark",
        height=420,
        paper_bgcolor="#1a1a2e",
        plot_bgcolor="#16213e",
        font_color="#ffffff",
    )
    fig.write_html(str(GOLD_SENT_DIR / "sentiment_validation.html"))
    print("✅ sentiment_validation.html salvo")
    try:
        fig.show()
    except (ValueError, Exception) as e:
        print(f"   (Visualização interativa não disponível no ambiente: {type(e).__name__})")

except ImportError:
    print("⚠️  Plotly não disponível — instale para visualizações")


✅ sentiment_validation.html salvo
   (Visualização interativa não disponível no ambiente: ValueError)


## 📋 Seção 8 — Resumo de Outputs

In [9]:

print("\n📋 OUTPUTS NB05")
print("=" * 65)
outputs = [
    (SILVER_DIR / "silver_enriched.parquet",          "Silver + sentimento + sinais (input NB06/NB07)"),
    (GOLD_SENT_DIR / "articles_sentiment.parquet",    "Docs com scores e flags (input NB06/NB07)"),
    (GOLD_SENT_DIR / "sentiment_by_source.parquet",   "Stats de sentimento por fonte"),
    (GOLD_SENT_DIR / "sentiment_by_month.parquet",    "Stats de sentimento por mês"),
    (GOLD_SENT_DIR / "sentiment_validation.html",     "Visualização Plotly (validação)"),
]
for p, desc in outputs:
    if p.exists():
        sz = p.stat().st_size // 1024
        print(f"  ✅ {p.name:<46} {sz:>5} KB — {desc}")
    else:
        print(f"  ⚠️  {p.name} — não gerado (verifique dados Silver)")

log_quality_check(
    logger, nb="NB05",
    total_docs=len(df_enriched),
    positivo=int((df_enriched["sentiment_label"]=="positivo").sum()),
    negativo=int((df_enriched["sentiment_label"]=="negativo").sum()),
    neutro=int((df_enriched["sentiment_label"]=="neutro").sum()),
)


📋 OUTPUTS NB05
  ✅ silver_enriched.parquet                          327 KB — Silver + sentimento + sinais (input NB06/NB07)
  ✅ articles_sentiment.parquet                        42 KB — Docs com scores e flags (input NB06/NB07)
  ✅ sentiment_by_source.parquet                        6 KB — Stats de sentimento por fonte
  ✅ sentiment_by_month.parquet                         3 KB — Stats de sentimento por mês
  ✅ sentiment_validation.html                       4741 KB — Visualização Plotly (validação)
2026-06-14T15:49:32 | INFO     | fii_pipeline.ingestion | QUALITY_CHECK | nb=NB05 | total_docs=126 | positivo=70 | negativo=17 | neutro=39



## ✅ NB05 Completo

| Artefato | Localização | Consumido por |
|----------|-------------|---------------|
| `silver_enriched.parquet` | `data/silver/` | NB06, NB07 |
| `articles_sentiment.parquet` | `data/gold/sentiment/` | NB06, NB07 |
| `sentiment_by_source.parquet` | `data/gold/sentiment/` | NB07 |
| `sentiment_by_month.parquet` | `data/gold/sentiment/` | NB07 |

▶️ **Próximo:** `06_marketing_intelligence.ipynb`